# Reprodução do FETCH — Etapa 2

Este notebook reproduz o experimento de Agarwal et al. (2020) — *FETCH: A deep-learning based classifier for fast transient classification* — no Google Colab.

**Compatibilidade Keras:** a partir do TF 2.16, Keras 3 é o padrão. Como o FETCH usa a API do Keras 2, instalamos `tf_keras` e definimos `TF_USE_LEGACY_KERAS=1` antes de qualquer `import tensorflow` — conforme [documentação oficial do Keras](https://keras.io/getting_started/).

**Geração de candidatos:** o branch `tf2` usa `your_candmaker.py` (da biblioteca `your`), não o `candmaker.py` legado.

**Instalação:** usamos `python setup.py install` (não `pip install -e .`) para que `predict.py` e `train.py` sejam registrados no PATH.

**Estratégia em duas fases:**
1. **Smoke test** (Seção 3): 3 candidatos de exemplo — valida o pipeline antes de gastar GPU com o dataset completo.
2. **Avaliação completa** (Seção 5): dataset real do artigo — gera as métricas que irão para o artigo.

**Ambiente:** Google Colab com GPU T4 (Runtime → Change runtime type → GPU).

## 1. Configuração do ambiente

In [ ]:
# Verificar GPU disponível
!nvidia-smi

Tue Jun 30 23:16:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# TF_USE_LEGACY_KERAS deve ser definido ANTES de qualquer 'import tensorflow'
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

!pip install -q tf_keras
!pip install -q scikit-learn pandas numpy matplotlib seaborn h5py tqdm pyyaml

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import tensorflow as tf
import tf_keras
print('TensorFlow:', tf.__version__)
print('tf_keras (Keras 2):', tf_keras.__version__)
print('GPU disponível:', tf.config.list_physical_devices('GPU'))

# Sanidade: tf.keras deve vir de tf_keras, não de keras 3
assert 'tf_keras' in tf.keras.layers.Layer.__module__, (
    'tf.keras ainda aponta para Keras 3! '
    'Reinicie o runtime (Runtime > Restart session) e execute desde a primeira célula.'
)
print('OK — tf.keras está corretamente resolvendo para Keras 2 (tf_keras).')

TensorFlow: 2.20.0
tf_keras (Keras 2): 2.20.0
GPU disponível: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
OK — tf.keras está corretamente resolvendo para Keras 2 (tf_keras).


## 2. Clonar e instalar o repositório FETCH (branch tf2)

In [ ]:
# 'python setup.py install' (não 'pip install -e .') é necessário para registrar
# predict.py e train.py no PATH do sistema
%cd /content
!git clone -b tf2 https://github.com/devanshkv/fetch.git
%cd /content/fetch
!python setup.py install -q

/content
fatal: destination path 'fetch' already exists and is not an empty directory.
/content/fetch
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'tests_require'
  warnings.warn(msg)
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

   

In [ ]:
# Verificar se predict.py está no PATH
!which predict.py && predict.py --help | head -5

/usr/local/bin/predict.py
/usr/local/bin/predict.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  __import__('pkg_resources').run_script('fetch==0.2.0', 'predict.py')
usage: predict.py [-h] [-v] [-g GPU_ID] [-n NPROC] -c DATA_DIR [-b BATCH_SIZE]
                  -m MODEL [-p PROBABILITY]

Fast Extragalactic Transient Candiate Hunter (FETCH)



## 3. Instalar a biblioteca `your` (candmaker do branch tf2)

In [ ]:
# No branch tf2, a geração de candidatos usa your_candmaker.py (da lib 'your'),
# não o candmaker.py legado
!pip install -q your
!which your_candmaker.py && your_candmaker.py --help | head -5

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
/usr/local/bin/your_candmaker.py
usage: your_candmaker.py [-h] [-v] [-fs FREQUENCY_SIZE]
                         [-g GPU_ID [GPU_ID ...]] [-ts TIME_SIZE] -c
                         CAND_PARAM_FILE [-n NPROC] [-o FOUT] [-r]
                         [-sksig SPECTRAL_KURTOSIS_SIGMA]
                         [-sgsig SAVGOL_SIGMA] [-sgfw SAVGOL_FREQUENCY_WINDOW]


## 4. Smoke test — validação do pipeline com os 3 candidatos de exemplo

Roda o pipeline completo nos 3 filterbanks de exemplo do próprio README para confirmar que o ambiente está correto.
**Não gera métricas estatisticamente válidas (n=3)** — serve apenas como checagem antes de avançar para o dataset completo.

⚠️ **Ação necessária:** abra o [README do repositório](https://github.com/devanshkv/fetch/tree/tf2), localize a frase *'Test filterbank data can be downloaded from here'*, copie o link real e substitua `<URL_DOS_FILTERBANKS_DE_EXEMPLO>` abaixo.

### 4.1 Download dos filterbanks de exemplo

In [ ]:
os.makedirs('/content/data_smoke', exist_ok=True)
%cd /content/data_smoke
!wget -q --show-progress <URL_DOS_FILTERBANKS_DE_EXEMPLO>
!ls -lh

/content/data_smoke
/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `wget -q --show-progress <URL_DOS_FILTERBANKS_DE_EXEMPLO>'
total 8.0K
drwxr-xr-x 2 root root 4.0K Jun 30 22:52 candidates
-rw-r--r-- 1 root root  183 Jun 30 22:52 cands.csv


### 4.2 Criação do cands.csv e geração dos candidatos h5

O README confirma que `your_candmaker.py` gera exatamente 3 arquivos `.h5` a partir desses 3 filterbanks, com os parâmetros abaixo.
Formato do CSV: `fil,snr,tstart,dm,width,label,kill_mask`

In [ ]:
cands_csv = """/content/data_smoke/28.fil,18.66470,2.0288800,475.28400,4,1,
/content/data_smoke/29.fil,16.81280,2.0288800,475.28400,4,1,
/content/data_smoke/34.fil,13.92710,2.0288800,475.28400,4,1,
"""
with open('/content/data_smoke/cands.csv','w') as f:
    f.write(cands_csv)

os.makedirs('/content/data_smoke/candidates', exist_ok=True)
!your_candmaker.py -fs 256 -ts 256 \
    -c /content/data_smoke/cands.csv \
    -o /content/data_smoke/candidates/

!ls /content/data_smoke/candidates/

/usr/local/bin/your_candmaker.py:278: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  + datetime.utcnow().strftime("your_candmaker_%Y_%m_%d_%H_%M_%S_%f.log")
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'file'

The above exception was the direct cause of the following exception:

Tr

### 4.3 Predict com modelo `a` (smoke test)

In [ ]:
# predict.py salva results_<modelo>.csv no diretório de trabalho atual
%cd /content/data_smoke
!predict.py -c /content/data_smoke/candidates/ -m a

/content/data_smoke
/usr/local/bin/predict.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  __import__('pkg_resources').run_script('fetch==0.2.0', 'predict.py')
2026-06-30 23:18:04,589 - <module> -root - INFO - Using multiprocessing with 4 workers
2026-06-30 23:18:04,593 - get_model -fetch.utils - INFO - Getting model a
2026-06-30 23:18:06.445310: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1782861486.446831   27039 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-30 23:18:19,818 - <module> -__main__ - WARNING - No candidates to evaluate in directory: /content/data_smoke/candidates/


In [ ]:
import subprocess, os

# Ver onde o CSV foi parar
result = subprocess.run(['find', '/content', '-name', 'results_*.csv'],
                        capture_output=True, text=True)
print("CSVs encontrados:", result.stdout)

# Ver o diretório de trabalho atual
print("CWD atual:", os.getcwd())

# Listar o diretório /content/data_smoke completo
print("\nConteúdo de /content/data_smoke:")
for root, dirs, files in os.walk('/content/data_smoke'):
    for f in files:
        print(os.path.join(root, f))

CSVs encontrados: 
CWD atual: /content/data_smoke

Conteúdo de /content/data_smoke:
/content/data_smoke/cands.csv
/content/data_smoke/candidates/your_candmaker_2026_06_30_22_03_51_128848.log
/content/data_smoke/candidates/your_candmaker_2026_06_30_23_17_56_596562.log
/content/data_smoke/candidates/your_candmaker_2026_06_30_22_52_07_646530.log


In [ ]:
!cat /content/data_smoke/candidates/your_candmaker_2026_06_30_22_03_51_128848.log

2026-06-30 22:03:51,129 - <module> -root - INFO - Input Arguments:-
2026-06-30 22:03:51,129 - <module> -root - INFO - cand_param_file: '/content/data_smoke/cands.csv'
2026-06-30 22:03:51,129 - <module> -root - INFO - flag_rfi: False
2026-06-30 22:03:51,129 - <module> -root - INFO - fout: '/content/data_smoke/candidates/'
2026-06-30 22:03:51,129 - <module> -root - INFO - frequency_size: 256
2026-06-30 22:03:51,129 - <module> -root - INFO - gpu_id: [-1]
2026-06-30 22:03:51,129 - <module> -root - INFO - no_log_file: False
2026-06-30 22:03:51,129 - <module> -root - INFO - nproc: 2
2026-06-30 22:03:51,129 - <module> -root - INFO - opt_dm: False
2026-06-30 22:03:51,129 - <module> -root - INFO - savgol_frequency_window: 15
2026-06-30 22:03:51,129 - <module> -root - INFO - savgol_sigma: 4
2026-06-30 22:03:51,129 - <module> -root - INFO - spectral_kurtosis_sigma: 4
2026-06-30 22:03:51,129 - <module> -root - INFO - time_size: 256
2026-06-30 22:03:51,129 - <module> -root - INFO - verbose: False
2

In [ ]:
!cat /content/data_smoke/cands.csv
!echo "---"
!head -1 /content/data_smoke/cands.csv | cat -A  # mostra caracteres invisíveis

/content/data_smoke/28.fil,18.66470,2.0288800,475.28400,4,1,
/content/data_smoke/29.fil,16.81280,2.0288800,475.28400,4,1,
/content/data_smoke/34.fil,13.92710,2.0288800,475.28400,4,1,
---
/content/data_smoke/28.fil,18.66470,2.0288800,475.28400,4,1,$


In [ ]:
!ls -lh /content/data_smoke/*.fil 2>/dev/null || echo "NENHUM .fil encontrado"

NENHUM .fil encontrado


In [ ]:
import os
for root, dirs, files in os.walk('/content/data_smoke'):
    for f in files:
        print(os.path.join(root, f))

/content/data_smoke/cands.csv
/content/data_smoke/candidates/your_candmaker_2026_06_30_22_03_51_128848.log
/content/data_smoke/candidates/your_candmaker_2026_06_30_23_17_56_596562.log
/content/data_smoke/candidates/your_candmaker_2026_06_30_22_52_07_646530.log


In [ ]:
!wget -q --spider http://astro.phys.wvu.edu/fetch/ 2>&1
!curl -I http://astro.phys.wvu.edu/fetch/ 2>&1 | head -5

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0   729    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
HTTP/1.1 200 OK
Server: nginx/1.14.1


In [ ]:
!wget -q -O - http://astro.phys.wvu.edu/fetch/ 2>&1 | grep -i "href\|\.tar\|\.h5\|\.zip\|\.gz"

<a href="train_set/train_data.hdf5" download="train_set/train_data.hdf5">Download</a>
<a href="test_set/test_data.hdf5" download="test_set/test_data.hdf5">Download</a>


In [ ]:
!curl -sI http://astro.phys.wvu.edu/fetch/test_set/test_data.hdf5 | grep -i content-length
!curl -sI http://astro.phys.wvu.edu/fetch/train_set/train_data.hdf5 | grep -i content-length

Content-Length: 6036831364
Content-Length: 17653183539


In [ ]:
!predict.py --help

/usr/local/bin/predict.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  __import__('pkg_resources').run_script('fetch==0.2.0', 'predict.py')
usage: predict.py [-h] [-v] [-g GPU_ID] [-n NPROC] -c DATA_DIR [-b BATCH_SIZE]
                  -m MODEL [-p PROBABILITY]

Fast Extragalactic Transient Candiate Hunter (FETCH)

options:
  -h, --help            show this help message and exit
  -v, --verbose         Be verbose (default: False)
  -g GPU_ID, --gpu_id GPU_ID
                        GPU ID (use -1 for CPU) (default: 0)
  -n NPROC, --nproc NPROC
                        Number of processors for training (default: 4)
  -c DATA_DIR, --data_dir DATA_DIR
                        Directory with candidate h5s. (default: None)
  -b BATCH_SIZE, --batch_size BATCH_SIZE
                        Batch size for training data (default: 8)
  -m MODEL, --model MODEL
                        Index of the model to train (default:

In [ ]:
import os
os.makedirs('/content/data_full', exist_ok=True)
%cd /content/data_full
!wget -q --show-progress http://astro.phys.wvu.edu/fetch/test_set/test_data.hdf5

/content/data_full
test_data.hdf5.2     17%[==>                 ]   1.01G  14.8MB/s    in 76s     
test_data.hdf5.2     35%[+++===>             ]   2.02G  8.76MB/s    in 2m 0s   
test_data.hdf5.2     53%[+++++++==>          ]   3.03G  12.9MB/s    in 80s     
test_data.hdf5.2     71%[++++++++++===>      ]   4.04G  14.1MB/s    in 76s     
test_data.hdf5.2     89%[++++++++++++++==>   ]   5.04G  13.4MB/s    in 81s     
test_data.hdf5.2    100%[+++++++++++++++++==>]   5.62G  14.7MB/s    in 43s     


In [ ]:
!pip install -q h5py
# Célula 2 — inspecionar estrutura
import h5py

with h5py.File('/content/data_full/test_data.hdf5', 'r') as f:
    print("Chaves raiz:", list(f.keys()))
    first_key = list(f.keys())[0]
    print(f"\nPrimeira chave: '{first_key}'")
    item = f[first_key]
    if hasattr(item, 'keys'):
        print("Subchaves:", list(item.keys()))
        for k in list(item.keys())[:3]:
            print(f"  {k}: shape={item[k].shape}, dtype={item[k].dtype}")
    else:
        print("Shape:", item.shape, "dtype:", item.dtype)
    print(f"\nTotal de entradas no arquivo: {len(f.keys())}")

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Chaves raiz: ['data_dm_time', 'data_freq_time', 'data_labels']

Primeira chave: 'data_dm_time'
Shape: (13983, 256, 256, 1) dtype: float32

Total de entradas no arquivo: 3


In [ ]:
import h5py

# Ver rótulos e confirmar estrutura completa
with h5py.File('/content/data_full/test_data.hdf5', 'r') as f:
    print("data_labels shape:", f['data_labels'].shape)
    print("data_labels dtype:", f['data_labels'].dtype)
    print("Primeiros 10 rótulos:", f['data_labels'][:10])
    print("Positivos (FRB):", int((f['data_labels'][:] == 1).sum()))
    print("Negativos (RFI):", int((f['data_labels'][:] == 0).sum()))

# Ver formato de um .h5 individual que o predict.py já gerou (do smoke test ou qualquer outro)
# Se não tiver nenhum .h5 individual disponível, vamos inspecionar o código-fonte do predict.py
!grep -n "h5\|hdf5\|keys\|data_dm\|data_freq\|label" /content/fetch/fetch/utils.py 2>/dev/null | head -30
!grep -n "h5\|hdf5\|keys\|data_dm\|data_freq\|label" /usr/local/lib/python3.12/dist-packages/fetch/utils.py 2>/dev/null | head -30

data_labels shape: (13983,)
data_labels dtype: bool
Primeiros 10 rótulos: [False False False False False False False False False False]
Positivos (FRB): 6664
Negativos (RFI): 7319


In [ ]:
import subprocess
result = subprocess.run(['find', '/usr/local/lib', '-name', 'utils.py', '-path', '*/fetch/*'],
                       capture_output=True, text=True)
print(result.stdout)

# Verificar como o fetch está instalado
!pip show fetch
!python -c "import pkg_resources; print(pkg_resources.require('fetch')[0].location)"

/usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg/fetch/utils.py

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Name: fetch
Version: 0.2.0
Summary: FETCH (Fast Extragalactic Transient Candidate Hunter)
Home-page: https://github.com/devanshkv/fetch
Author: ['Devansh Agarwal', 'Kshitij Aggarwal']
Author-email: ['devansh.kv@gmail.com', 'ka0064@mix.wvu.edu']
License: GNU General Public License v3.0
Location: /usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg
Requires: 
Required-by: 
<string>:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
/usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg


In [ ]:
# Verificar se os pesos já foram baixados pelo predict.py anteriormente
!find /root -name "*.h5" -path "*fetch*" 2>/dev/null | head -10
!find /tmp -name "*.h5" 2>/dev/null | head -10
!find ~/.keras -name "*.h5" 2>/dev/null | head -10

/root/.keras/models/a_ft_DenseNet121_2_dt_Xception_13_256.h5


In [ ]:
import sys
sys.path.insert(0, '/usr/local/lib/python3.12/dist-packages/fetch-0.2.0-py3.12.egg')

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import h5py
import numpy as np
import pandas as pd
import time
from fetch.utils import get_model
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

# Carregar dataset de teste
print("Carregando test_data.hdf5...")
with h5py.File('/content/data_full/test_data.hdf5', 'r') as f:
    dm_time   = f['data_dm_time'][:]    # (13983, 256, 256, 1)
    freq_time = f['data_freq_time'][:]  # (13983, 256, 256, 1)
    labels    = f['data_labels'][:].astype(int)  # (13983,)

print(f"Dataset carregado: {len(labels)} candidatos ({labels.sum()} FRBs, {(labels==0).sum()} RFIs)")

# Avaliar os 11 modelos
modelos = list('abcdefghijk')
resumo  = []
os.makedirs('/content/results', exist_ok=True)

for m in modelos:
    print(f"\n=== Modelo {m} ===")
    t0 = time.time()

    model = get_model(m)

    # predict retorna probabilidade de ser FRB
    probs = model.predict([freq_time, dm_time], batch_size=32, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)

    # Salvar CSV individual (mesmo formato do predict.py original)
    df_m = pd.DataFrame({
        'candidate': [f'cand_{i:05d}' for i in range(len(labels))],
        'probability': probs,
        'label': labels
    })
    df_m.to_csv(f'/content/results/results_{m}.csv', index=True)

    acc  = accuracy_score(labels, preds)
    rec  = recall_score(labels, preds, zero_division=0)
    prec = precision_score(labels, preds, zero_division=0)
    f1   = f1_score(labels, preds, zero_division=0)
    t    = time.time() - t0

    resumo.append({'modelo': m, 'acuracia': acc, 'recall': rec,
                   'precisao': prec, 'f1_score': f1, 'tempo_s': t})
    print(f"  Acurácia: {acc:.4f} | Recall: {rec:.4f} | Precisão: {prec:.4f} | F1: {f1:.4f} | {t:.1f}s")

df_resumo = pd.DataFrame(resumo)
df_resumo.to_csv('/content/results/resumo_metricas.csv', index=False)
print("\n=== RESUMO FINAL ===")
print(df_resumo.to_string(index=False))

Carregando test_data.hdf5...
Dataset carregado: 13983 candidatos (6664 FRBs, 7319 RFIs)

=== Modelo a ===


In [ ]:
import pandas as pd
df_smoke = pd.read_csv('/content/data_smoke/results_a.csv')
print("Smoke test concluído. Resultado do modelo 'a' nos 3 candidatos de exemplo:")
print(df_smoke)

assert len(df_smoke) == 3, 'Esperado 3 candidatos no smoke test!'
print('\nCheckpoint OK — pipeline funcional. Prossiga para a Seção 5.')

## 5. Download do dataset de teste completo

O conjunto de teste real do artigo (Tabela 1 de Agarwal et al., 2020) contém 6.664 candidatos positivos e 7.319 negativos.
Os dados estão disponíveis em `astro.phys.wvu.edu/fetch`.

⚠️ **Ação necessária:** confirme a estrutura de arquivos no servidor e substitua `<URL_DATASET_COMPLETO>` abaixo.

In [ ]:
os.makedirs('/content/data_full', exist_ok=True)
%cd /content/data_full
!wget -q --show-progress <URL_DATASET_COMPLETO>
!ls -lh

In [ ]:
os.makedirs('/content/data_full/candidates', exist_ok=True)
!your_candmaker.py -fs 256 -ts 256 \
    -c /content/data_full/cands_full.csv \
    -o /content/data_full/candidates/

## 6. Avaliação completa dos 11 modelos

In [ ]:
import time

modelos = list('abcdefghijk')
tempos_execucao = {}

os.makedirs('/content/results', exist_ok=True)
# predict.py salva results_<modelo>.csv no diretório de trabalho atual
%cd /content/results

for m in modelos:
    print(f'\n=== Avaliando modelo {m} ===')
    t0 = time.time()
    !predict.py -c /content/data_full/candidates/ -m {m}
    tempos_execucao[m] = time.time() - t0
    print(f'Modelo {m} concluído em {tempos_execucao[m]:.1f}s')

## 7. Consolidação dos resultados

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

resumo = []
for m in modelos:
    df = pd.read_csv(f'/content/results/results_{m}.csv')
    y_true = df['label']
    y_pred = (df['probability'] >= 0.5).astype(int)
    resumo.append({
        'modelo': m,
        'acuracia':  accuracy_score(y_true, y_pred),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'precisao':  precision_score(y_true, y_pred, zero_division=0),
        'f1_score':  f1_score(y_true, y_pred, zero_division=0),
        'tempo_s':   tempos_execucao[m]
    })

df_resumo = pd.DataFrame(resumo)
df_resumo.to_csv('/content/results/resumo_metricas.csv', index=False)
df_resumo

## 8. Comparação com a Tabela 4 do artigo original

Valores extraídos diretamente de Agarwal et al. (2020), MNRAS 497(2):1661–1674, Tabela 4.

In [ ]:
valores_originais = {
    'modelo':            list('abcdefghijk'),
    'acuracia_original': [99.88,99.86,99.86,99.86,99.85,99.81,99.79,99.76,99.75,99.68,99.66],
    'recall_original':   [99.92,99.92,99.78,99.78,99.75,99.70,99.59,99.72,99.59,99.59,99.62],
    'fscore_original':   [99.87,99.85,99.85,99.85,99.84,99.79,99.77,99.74,99.73,99.65,99.63],
}
df_orig = pd.DataFrame(valores_originais)
df_orig['acuracia_original'] /= 100
df_orig['recall_original']   /= 100
df_orig['fscore_original']   /= 100

df_comp = df_resumo.merge(df_orig, on='modelo')
df_comp['delta_acuracia'] = df_comp['acuracia'] - df_comp['acuracia_original']
df_comp['delta_recall']   = df_comp['recall']   - df_comp['recall_original']
df_comp['delta_fscore']   = df_comp['f1_score']  - df_comp['fscore_original']
df_comp

## 9. Visualizações

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_resumo.set_index('modelo')[['acuracia','recall','precisao','f1_score']].plot(
    kind='bar', ax=axes[0], rot=0)
axes[0].set_title('Métricas por modelo — FETCH (reprodução)')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right')

df_resumo.set_index('modelo')['tempo_s'].plot(kind='bar', ax=axes[1], color='salmon', rot=0)
axes[1].set_title('Tempo de execução por modelo (s)')
axes[1].set_ylabel('Segundos')

plt.tight_layout()
plt.savefig('/content/results/metricas_por_modelo.png', dpi=200)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()
for i, m in enumerate(modelos):
    df = pd.read_csv(f'/content/results/results_{m}.csv')
    y_true = df['label']
    y_pred = (df['probability'] >= 0.5).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False)
    axes[i].set_title(f'Modelo {m}')
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')
axes[-1].axis('off')
plt.tight_layout()
plt.savefig('/content/results/matrizes_confusao.png', dpi=200)
plt.show()

## 10. Download dos artefatos e registro do ambiente

In [ ]:
from google.colab import files
files.download('/content/results/resumo_metricas.csv')
files.download('/content/results/metricas_por_modelo.png')
files.download('/content/results/matrizes_confusao.png')

In [ ]:
!pip freeze > /content/results/environment_freeze.txt
!python --version >> /content/results/environment_freeze.txt
!nvidia-smi >> /content/results/environment_freeze.txt
files.download('/content/results/environment_freeze.txt')